In [ ]:
from langgraph.graph import StateGraph,START,END,add_messages
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

In [2]:
load_dotenv()

True

In [ ]:
class ChatState(TypedDict):
    messages:Annotated [list[BaseMessage],add_messages]
    # here we can also use list[str] but all human message , ai message, system message, tool message inherit from base msg  
    # if we dont use reducer previous msgs will be replaced
    # we can use , operator.add but add message is more efficient to add base messages in langgraph

In [9]:
llm= ChatOpenAI()

In [11]:
def chat_node_python(state:ChatState):
    messages= state['messages']
    response= llm.invoke(messages)
    return {'messages':[response]}


In [ ]:
checkpointer=MemorySaver()
graph= StateGraph(ChatState)


graph.add_node('chat_node',chat_node_python)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot=graph.compile(checkpointer=checkpointer)



In [ ]:
#intial_state={
   # 'messages': [HumanMessage(content='what is the capital of india')]
#}
#chatbot.invoke(intial_state)

In [ ]:
# to make continous chat bot 
thread_id='1'
while True:
    user_message=input('Type here :')
    print ('User:',user_message)
    if user_message.strip().lower() in ['exit','bye','quit']:
        break
    config = {'configurable':{'thread_id':thread_id}}
    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)
    print('AI:', response['messages'][-1].content)#-1 gives last message
    